# Seaborn `FacetGrid` — siatki wykresów wg kategorii

**Problem:** `FacetGrid` dzieli dane na podwykresy według wartości kolumn kategorycznych (`col=`, `row=`, `hue=`) i odpala tę samą funkcję rysującą na każdym podzbiorze. To potężne narzędzie do szybkiego porównania rozkładów/zależności między grupami, ale ma kilka nieoczywistych zasad (`.map()` vs `.map_dataframe()`, ręczne `add_legend()`) i pułapek, które łatwo przeoczyć.

**Porównanie:**
- `FacetGrid` — niskopoziomowe, elastyczne narzędzie: Ty decydujesz, jaką funkcję odpalić na każdym facecie.
- `sns.relplot()` / `sns.displot()` / `sns.catplot()` — wysokopoziomowe funkcje *figure-level*, które **wewnętrznie używają `FacetGrid`**, ale mają gotowy, prostszy interfejs (`kind=` zamiast ręcznego `.map()`).

**Kiedy stosować:** `relplot`/`displot`/`catplot`, gdy standardowy wykres (scatter, histogram, boxplot...) w pełni wystarcza — mniej kodu. `FacetGrid` bezpośrednio, gdy potrzebujesz customowej funkcji rysującej, kilku nakładających się warstw na facet, albo funkcji spoza standardowego zestawu seaborn. Więcej w Sekcji 7.

**Uwaga o stylu:** notatka celowo nie narzuca customowych motywów/palet (`sns.set_theme()`, własne palety kolorów) — zostają domyślne ustawienia matplotlib/seaborn, żeby styling było łatwo dostosować samodzielnie pod konkretny raport.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

rng = np.random.default_rng(7)
n = 400

region = rng.choice(["North", "South", "East", "West"], n)
category = rng.choice(["A", "B", "C"], n)
channel = rng.choice(["Online", "Retail"], n)

# Sztuczne, ale sensowne różnice między grupami - żeby wykresy faktycznie coś pokazywały
category_effect = {"A": 1300, "B": 950, "C": 700}
region_effect = {"North": 100, "South": 0, "East": -50, "West": 150}

sales = np.array([category_effect[c] + region_effect[r] for c, r in zip(category, region)]) + rng.normal(0, 150, n)
units = np.clip((sales / 80 + rng.normal(0, 3, n)).round(), 1, None)

df = pd.DataFrame({"region": region, "category": category, "channel": channel, "sales": sales, "units": units})
df.head()

## Sekcja 1 — Podstawy: tworzenie `FacetGrid` i `.map()`

Dwa kroki zawsze w tej kolejności: (1) `sns.FacetGrid(dane, col=..., row=..., hue=...)` tworzy "pustą" siatkę osi, (2) `.map(funkcja, "kolumna")` odpala tę funkcję na każdym facecie osobno, z odpowiednim podzbiorem danych.

In [ ]:
g = sns.FacetGrid(df, col="category")
g.map(sns.histplot, "sales")

## Sekcja 2 — `.map()` vs `.map_dataframe()`

- **`.map(funkcja, "kolA", "kolB", ...)`** — wyciąga wskazane kolumny jako osobne tablice i przekazuje je *pozycyjnie* do funkcji (np. `plt.scatter(x, y)`). Działa dobrze z funkcjami matplotlib i prostymi funkcjami seaborn.
- **`.map_dataframe(funkcja, x="kolA", y="kolB", hue="kolC", ...)`** — przekazuje **cały podzbiór DataFrame** danego faceta jako `data=`, a nazwy kolumn jako kwargs. Wymagane, gdy funkcja sama chce znać nazwy kolumn (np. `sns.scatterplot(data=..., hue=...)` z własną logiką kolorowania) albo gdy potrzebujesz więcej niż `hue` przekazane przez `FacetGrid`.

In [ ]:
# .map() - pozycyjne kolumny
g = sns.FacetGrid(df, col="category")
g.map(plt.scatter, "sales", "units")

In [ ]:
# .map_dataframe() - cały podzbiór danych, funkcja sama zarządza hue
g = sns.FacetGrid(df, col="category")
g.map_dataframe(sns.scatterplot, x="sales", y="units", hue="channel")
g.add_legend()

## Sekcja 3 — `hue=` i ręczne `add_legend()`

`hue=` w konstruktorze `FacetGrid` koloruje warstwy w obrębie każdego faceta wg dodatkowej kolumny kategorycznej. **Legenda nie pojawia się automatycznie** — wymaga jawnego `.add_legend()` (patrz też Pułapka 4).

In [ ]:
g = sns.FacetGrid(df, col="region", hue="category", height=3)
g.map(sns.kdeplot, "sales", fill=True, alpha=0.4)
g.add_legend()

## Sekcja 4 — `col_wrap` i jednoczesne `row=` + `col=`

`col_wrap=N` zawija facety do N na wiersz — przydatne przy kolumnie z wieloma kategoriami, żeby nie robić jednego bardzo szerokiego rzędu. `row=` + `col=` naraz tworzy pełną siatkę dwuwymiarową (patrz Pułapka 2: te dwie opcje się wykluczają).

In [ ]:
g = sns.FacetGrid(df, col="region", col_wrap=2, height=3)
g.map(sns.histplot, "sales")

In [ ]:
g = sns.FacetGrid(df, row="channel", col="region", height=2.5)
g.map(sns.histplot, "sales")

## Sekcja 5 — `sharex`/`sharey` i rozmiar (`height`/`aspect`)

Domyślnie wszystkie facety mają wspólną skalę osi (`sharex=True`, `sharey=True`) — to ułatwia porównanie wartości między facetami, ale może spłaszczyć rozkład w facecie o mniejszym zakresie danych. `sharey=False` daje każdemu facetowi własną skalę — czytelniejszy kształt rozkładu, ale utrudnia bezpośrednie porównanie wysokości słupków.

`FacetGrid` nie przyjmuje `figsize` — rozmiar całej figury wynika z `height` (wysokość jednego faceta w calach) i `aspect` (stosunek szerokości do wysokości), pomnożonych przez liczbę facetów.

In [ ]:
g = sns.FacetGrid(df, col="category", sharey=False, height=3, aspect=1.2)
g.map(sns.histplot, "sales")

## Sekcja 6 — Customizacja: tytuły, etykiety osi, dostęp do `g.axes`

`set_titles()` i `set_axis_labels()` nadpisują domyślne, automatycznie generowane opisy. `g.axes.flat` daje bezpośredni dostęp do obiektów `matplotlib.Axes` każdego faceta — przydatne do doklejenia czegokolwiek, czego `FacetGrid` sam nie oferuje (np. linii referencyjnej powtórzonej na każdym podwykresie).

In [ ]:
g = sns.FacetGrid(df, col="category", height=3)
g.map(sns.histplot, "sales")
g.set_titles(col_template="Kategoria {col_name}")
g.set_axis_labels("Sprzedaż", "Liczba obserwacji")

In [ ]:
# Trik: linia średniej CAŁKOWITEJ (nie per-facet) na każdym podwykresie - do porównania grupy z ogółem
overall_mean = df["sales"].mean()

g = sns.FacetGrid(df, col="category", height=3)
g.map(sns.histplot, "sales")
for ax in g.axes.flat:
    ax.axvline(overall_mean, color="red", linestyle="--", linewidth=1)

## Sekcja 7 — Kiedy `FacetGrid`, kiedy `relplot`/`displot`/`catplot`

Te trzy funkcje *figure-level* budują `FacetGrid` za kulisami — `kind=` wybiera funkcję rysującą zamiast ręcznego `.map()`. Dla standardowych wykresów to krótszy zapis tego samego efektu.

In [ ]:
# To jest równoważne Sekcji 1 (histogram sprzedaży per kategoria) w jednej linii
sns.displot(data=df, x="sales", col="category", height=3)

## Sekcja 8 — Pułapki

### Pułapka 1 — funkcje figure-level (`relplot`/`displot`/`catplot`) ignorują `ax=`

Próba osadzenia `sns.relplot()` w istniejącym subplotcie przez `ax=` **nie rzuca wyjątku** — tylko cichy `UserWarning`. Efekt: przekazany `ax` zostaje całkowicie pusty, a funkcja i tak tworzy własną, osobną figurę. To wygląda jak zignorowany, mało istotny warning, a w praktyce oznacza pusty podwykres dokładnie tam, gdzie spodziewałeś się wykresu.

In [ ]:
import warnings

fig, ax = plt.subplots()
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    result = sns.relplot(data=df, x="sales", y="units", ax=ax)
    for w in caught:
        print(f"Ostrzeżenie: {w.message}")

print(f"\nCzy przekazany 'ax' dostał jakiekolwiek dane? {len(ax.collections) > 0}")
print(f"Czy relplot stworzył WŁASNĄ, osobną figurę? {result.figure is not fig}")
plt.close("all")

### Pułapka 2 — `col_wrap` i `row=` wzajemnie się wykluczają

`col_wrap` ma sens tylko przy jednowymiarowej siatce (`col=` bez `row=`). Próba połączenia obu daje jasny błąd — w przeciwieństwie do Pułapki 1, ta przynajmniej nie jest cicha.

In [ ]:
try:
    sns.FacetGrid(df, row="channel", col="region", col_wrap=2)
except ValueError as e:
    print(f"Błąd: {e}")

### Pułapka 3 — kolejność argumentów w `.map()` musi zgadzać się z sygnaturą funkcji

`.map(plt.scatter, "sales", "units")` podaje kolumny **pozycyjnie** — pierwsza staje się `x`, druga `y`. Zamiana kolejności zamienia miejscami osie bez żadnego ostrzeżenia, bo z punktu widzenia `plt.scatter` to poprawne wywołanie — tylko z innymi danymi niż zamierzone.

In [ ]:
g_correct = sns.FacetGrid(df, col="category", height=2.5)
g_correct.map(plt.scatter, "sales", "units")  # x=sales, y=units - zamierzone
first_x_correct = g_correct.axes.flat[0].collections[0].get_offsets()[0, 0]
plt.close(g_correct.fig)

g_swapped = sns.FacetGrid(df, col="category", height=2.5)
g_swapped.map(plt.scatter, "units", "sales")  # kolejność odwrócona - x i y zamienione
first_x_swapped = g_swapped.axes.flat[0].collections[0].get_offsets()[0, 0]
plt.close(g_swapped.fig)

print(f"Poprawna kolejność - pierwsza wartość x (powinno być 'sales', rząd setek/tysięcy): {first_x_correct:.1f}")
print(f"Zamieniona kolejność - pierwsza wartość x (teraz to 'units', rząd dziesiątek): {first_x_swapped:.1f}")

### Pułapka 4 — `hue=` bez `.add_legend()` = brak legendy w wyniku

Ustawienie `hue=` samo w sobie nie dodaje legendy do figury — trzeba to zrobić jawnie. Bez tego wykres pokazuje kolory, ale widz nie ma jak się dowiedzieć, co która oznacza.

In [ ]:
g = sns.FacetGrid(df, col="region", hue="category", height=2.5)
g.map(sns.kdeplot, "sales")
print(f"Legenda zaraz po .map(), przed add_legend(): {g.legend}")

g.add_legend()
print(f"Legenda po jawnym .add_legend(): {'obecna' if g.legend is not None else 'brak'}")
plt.close(g.fig)

## Podsumowanie

| Zadanie | Rozwiązanie |
|---|---|
| Podział na podwykresy wg jednej kolumny | `sns.FacetGrid(df, col="kolumna")` |
| Podział wg dwóch kolumn naraz (siatka 2D) | `col=` + `row=` |
| Zawijanie wielu kategorii do kilku na wiersz | `col_wrap=N` (wyklucza się z `row=`) |
| Kolorowanie warstw w obrębie faceta | `hue="kolumna"` + **jawne** `.add_legend()` |
| Odpalenie funkcji matplotlib/prostej funkcji seaborn | `.map(funkcja, "kolA", "kolB")` — argumenty pozycyjne |
| Odpalenie funkcji świadomej `DataFrame`/własnego `hue` | `.map_dataframe(funkcja, x=, y=, hue=, ...)` |
| Niezależna skala osi per facet | `sharex=False` / `sharey=False` |
| Rozmiar figury | `height=` (cale, wysokość faceta) × `aspect=` (proporcja) — nie `figsize` |
| Własne tytuły/etykiety | `.set_titles(col_template=...)`, `.set_axis_labels(...)` |
| Coś, czego `FacetGrid` nie oferuje wprost (np. linia referencyjna) | pętla po `g.axes.flat`, zwykłe metody `matplotlib.Axes` |
| Standardowy wykres bez ręcznego `.map()` | `sns.relplot`/`displot`/`catplot(..., col=..., kind=...)` |
| Zapis do pliku | `g.savefig("plik.png")` (na obiekcie `FacetGrid`, nie na `plt`) |

**Wniosek:** największe ryzyko to nie brak znajomości API, tylko dwie ciche pułapki — `ax=` ignorowany przez funkcje figure-level (tylko warning, pusty podwykres) i brakująca legenda przy `hue=`. Obie dają wynik, który *wygląda* na zadziałany, dopóki nie spojrzysz uważnie na figurę.